In [10]:
%%writefile TP1C_Servidor_Threads.py
import socket
import threading
from datetime import datetime

HOST = '127.0.0.1'
PORT = 6668
clientes_conectados = {} 

def manejar_cliente(conn, addr):
    inicio = datetime.now()
    clientes_conectados[addr] = inicio
    print(f"\n[+] Nuevo cliente: {addr}")
    print(f"Lista actual de clientes: {list(clientes_conectados.keys())}")

    try:
        conn.send(f"Conectado el {inicio.strftime('%H:%M:%S')}".encode())
        
        with open(f"recibido_{addr[1]}.txt", "wb") as f:
            while True:
                data = conn.recv(1024)
                if not data: 
                    break
                
                if b"FIN_ARCHIVO" in data:
                    datos_limpios = data.replace(b"FIN_ARCHIVO", b"")
                    if datos_limpios:
                        f.write(datos_limpios)
                    break
                
                f.write(data)
        
        duracion = datetime.now() - inicio
        respuesta = f"\nArchivo recibido. Estuviste conectado: {duracion.seconds} segundos."
        conn.send(respuesta.encode())
    finally:
        print(f"[-] Cliente desconectado: {addr}")
        del clientes_conectados[addr]
        conn.close()

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
sock.bind((HOST, PORT))
sock.listen(5)
print(f"Servidor Concurrente (Threads) escuchando en {PORT}...")

try:
    while True:
        conn, addr = sock.accept()
        threading.Thread(target=manejar_cliente, args=(conn, addr)).start()
except KeyboardInterrupt:
    print("\nCerrando servidor...")
finally:
    sock.close()

Overwriting TP1C_Servidor_Threads.py


In [1]:
%%writefile TP1C_Servidor_Select.py
import socket
import select
import sys
import time
from datetime import datetime

servidor = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
servidor.setblocking(0)
servidor.bind(('localhost', 10001))
servidor.listen(5)

entradas = [servidor]
tiempos = {} # {socket: tiempo_inicio}

print("Servidor Select escuchando en el puerto 10001...")

try:
    while entradas:
        readable, writable, exceptional = select.select(entradas, [], entradas)
        
        for s in readable:
            if s is servidor:
                conn, addr = s.accept()
                conn.setblocking(0)
                entradas.append(conn)
                tiempos[conn] = datetime.now()
                print(f"Nueva conexión: {addr}. Clientes totales: {len(entradas)-1}")
            else:
                data = s.recv(1024)
                if data:
                    print(f"Recibiendo datos de {s.getpeername()}")
                    # Simulación de respuesta de tiempo
                    ahora = datetime.now()
                    duracion = ahora - tiempos[s]
                    s.send(f"Hora conexión: {tiempos[s].strftime('%H:%M:%S')} | Duración: {duracion.seconds}s".encode())
                else:
                    entradas.remove(s)
                    s.close()
                    del tiempos[s]
except KeyboardInterrupt:
    print("Finalizando servidor...")
finally:
    servidor.close()

Writing TP1C_Servidor_Select.py


In [2]:
%%writefile TP1C_Cliente.py
import socket
import sys

def iniciar_cliente(puerto):
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        sock.connect(('localhost', puerto))
        
        # Recibir mensaje de bienvenida
        print(sock.recv(1024).decode())

        with open("archivo_prueba.txt", "w") as f:
            f.write("Este es el contenido del archivo enviado por el cliente.")
        
        with open("archivo_prueba.txt", "rb") as f:
            sock.sendall(f.read())
        
        # Señal de fin de archivo para el servidor de threads
        sock.send(b"FIN_ARCHIVO")
        
        # Recibir estadísticas de tiempo
        print("Respuesta del servidor:", sock.recv(1024).decode())

        input("\nPresione Enter para cerrar el cliente y limpiar recursos...")
    except Exception as e:
        print(f"Error: {e}")
    finally:
        sock.close()
        print("Socket cerrado. Cliente finalizado.")

if __name__ == "__main__":
    p = int(input("Ingrese puerto del servidor (6668 para Threads, 10001 para Select): "))
    iniciar_cliente(p)

Writing TP1C_Cliente.py


In [6]:
%%writefile TP1C_Cliente_Archivo.py
import socket
import time

# Configuracion
server_addr = ('localhost', 10001) # Puerto del Servidor Select
nombre_archivo = "mi_archivo.txt"

sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

try:
    sock.connect(server_addr)
    
    # Abrimos el archivo y mandamos su contenido
    with open(nombre_archivo, "rb") as f:
        contenido = f.read()
        print(f"Enviando el archivo '{nombre_archivo}'...")
        sock.sendall(contenido)
    
    # Recibimos la respuesta del servidor
    data = sock.recv(1024)
    print("Respuesta del servidor:", data.decode())
    while True:
        opcion = input("\n¿Desea cerrar la conexión y salir? (s/n): ").lower()
        if opcion == 's':
            print("Limpiando recursos...")
            break
        else:
            print("El cliente sigue conectado (esperando 5 segundos)...")
            time.sleep(5)

finally:
    sock.close()
    print("Conexión cerrada.")

Overwriting TP1C_Cliente_Archivo.py
